# CXR Triage — Clean Evaluation Notebook

**Model under evaluation:** DenseNet-121 + Focal Loss, no CLAHE, 224x224
(corrected logits pipeline)

Thresholds are derived from the VALIDATION set only, then frozen and
applied to the TEST set. No test-set information is used to pick
thresholds, avoiding data leakage.

In [ ]:
import sys
sys.path.append('D:/cxr-triage')

import torch
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    precision_recall_curve
)
from tqdm import tqdm

from src.data.dataset import ChestXrayDataset
from src.data.transforms import get_val_transforms
from src.models.densenet import DenseNetModel

LABELS = [
    'Atelectasis', 'Consolidation', 'Infiltration',
    'Pneumothorax', 'Edema', 'Emphysema', 'Fibrosis',
    'Effusion', 'Pneumonia', 'Pleural_Thickening',
    'Cardiomegaly', 'Nodule', 'Mass', 'Hernia'
]
CRITICAL = ['Pneumothorax', 'Edema', 'Pneumonia', 'Hernia']

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMAGE_ROOT = "F:/X ray dataset/Second Version"

# === Model under test ===
CHECKPOINT_PATH = "D:/cxr-triage/checkpoints/densenet_focal_fixed/best_model.pth"
IMAGE_SIZE = 224
USE_CLAHE = False
MODEL_LABEL = "DenseNet + Focal Loss (no CLAHE, 224)"

print(f"Device: {DEVICE}")
print(f"Evaluating: {MODEL_LABEL}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Image size: {IMAGE_SIZE}, CLAHE: {USE_CLAHE}")

## Sanity check 1 — zero patient leakage across splits

In [ ]:
train_df = pd.read_csv('D:/cxr-triage/data/processed/train.csv')
val_df = pd.read_csv('D:/cxr-triage/data/processed/val.csv')
test_df = pd.read_csv('D:/cxr-triage/data/processed/test.csv')

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

train_p = set(train_df['Patient ID'])
val_p = set(val_df['Patient ID'])
test_p = set(test_df['Patient ID'])

tv, tt, vt = len(train_p & val_p), len(train_p & test_p), len(val_p & test_p)
print(f"Train-Val overlap: {tv} | Train-Test overlap: {tt} | Val-Test overlap: {vt}")

assert tv == 0 and tt == 0 and vt == 0, "LEAKAGE DETECTED — STOP"
print("PASS: zero patient leakage")

## Sanity check 2 — label order matches training, model has no Sigmoid

In [ ]:
training_labels_order = [
    'Atelectasis', 'Consolidation', 'Infiltration',
    'Pneumothorax', 'Edema', 'Emphysema', 'Fibrosis',
    'Effusion', 'Pneumonia', 'Pleural_Thickening',
    'Cardiomegaly', 'Nodule', 'Mass', 'Hernia'
]
print("Label order match:", LABELS == training_labels_order)
assert LABELS == training_labels_order, "LABEL ORDER MISMATCH — STOP"

model = DenseNetModel(num_classes=14, pretrained=False).to(DEVICE)
checkpoint = torch.load(CHECKPOINT_PATH, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"\nCheckpoint epoch: {checkpoint['epoch']+1}")
print(f"Checkpoint train-time best AUC: {checkpoint['best_auc']:.4f}")

classifier = model.model.classifier
is_linear_only = isinstance(classifier, torch.nn.Linear)
print(f"\nClassifier: {classifier}")
print("PASS: classifier is plain Linear (no Sigmoid)" if is_linear_only
      else "FAIL: unexpected classifier structure")
assert is_linear_only, "SIGMOID DETECTED IN CLASSIFIER — STOP"

## Sanity check 3 — forward pass produces unbounded logits, no NaN/Inf

In [ ]:
sanity_transform = get_val_transforms(image_size=IMAGE_SIZE, use_clahe=USE_CLAHE)
sanity_dataset = ChestXrayDataset(csv_path=None, image_root=IMAGE_ROOT, transform=sanity_transform)
sanity_dataset.df = val_df.head(16).reset_index(drop=True)
sanity_loader = DataLoader(sanity_dataset, batch_size=16, shuffle=False)

with torch.no_grad():
    sample_images, sample_targets = next(iter(sanity_loader))
    sample_images = sample_images.to(DEVICE)
    sample_logits = model(sample_images)

lo, hi = sample_logits.min().item(), sample_logits.max().item()
has_nan = torch.isnan(sample_logits).any().item()
has_inf = torch.isinf(sample_logits).any().item()

print(f"Logit min: {lo:.4f} | Logit max: {hi:.4f}")
print(f"NaN: {has_nan} | Inf: {has_inf}")

bounded = (0.0 <= lo) and (hi <= 1.0)
print("PASS: unbounded raw logits, as expected" if not bounded
      else "FAIL: outputs look bounded — possible sigmoid bug")
assert not bounded, "LOGITS APPEAR BOUNDED — STOP"
assert not has_nan and not has_inf, "NaN/Inf DETECTED — STOP"

## Inference helper

In [ ]:
def get_predictions(df, batch_size=16):
    transform = get_val_transforms(image_size=IMAGE_SIZE, use_clahe=USE_CLAHE)
    dataset = ChestXrayDataset(csv_path=None, image_root=IMAGE_ROOT, transform=transform)
    dataset.df = df
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                         num_workers=4, pin_memory=True)

    all_logits, all_targets = [], []
    with torch.no_grad():
        for images, targets in tqdm(loader, desc="Inference"):
            images = images.to(DEVICE)
            with torch.amp.autocast('cuda'):
                logits = model(images)
            all_logits.append(logits.cpu().numpy())
            all_targets.append(targets.numpy())

    logits = np.concatenate(all_logits, axis=0).astype(np.float32)
    targets = np.concatenate(all_targets, axis=0).astype(np.float32)
    probs = 1 / (1 + np.exp(-logits))
    return probs, targets

print("Inference function ready")

## Run inference on VALIDATION set, find thresholds (leak-free)

In [ ]:
val_probs, val_targets = get_predictions(val_df)
print(f"Val shape: {val_probs.shape}")
in_range = (val_probs.min() >= 0.0) and (val_probs.max() <= 1.0)
print("PASS: probs bounded [0,1]" if in_range else "FAIL: probs out of range")
assert in_range

optimal_thresholds = {}
print(f"\n{'Finding':<22} {'Threshold':<12} {'Val F1':<10}")
print("-" * 50)
for i, label in enumerate(LABELS):
    precisions, recalls, thresh = precision_recall_curve(val_targets[:, i], val_probs[:, i])
    f1s = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-8)
    best_idx = np.argmax(f1s)
    optimal_thresholds[label] = float(thresh[best_idx])
    print(f"{label:<22} {thresh[best_idx]:<12.4f} {f1s[best_idx]:<10.4f}")
print("-" * 50)
print("Thresholds frozen — will be applied to TEST set only.")

## Run inference on TEST set, apply frozen thresholds, compute final metrics

In [ ]:
test_probs, test_targets = get_predictions(test_df)
print(f"Test shape: {test_probs.shape}")

test_pred = np.zeros_like(test_probs, dtype=np.float32)
for i, label in enumerate(LABELS):
    test_pred[:, i] = (test_probs[:, i] >= optimal_thresholds[label]).astype(np.float32)

triage_map = {
    'Pneumothorax': 'Critical', 'Edema': 'Critical',
    'Pneumonia': 'Critical', 'Hernia': 'Critical',
    'Consolidation': 'Urgent', 'Effusion': 'Urgent',
    'Cardiomegaly': 'Urgent', 'Mass': 'Urgent',
    'Atelectasis': 'Urgent', 'Nodule': 'Urgent',
    'Infiltration': 'Urgent', 'Emphysema': 'Routine',
    'Fibrosis': 'Routine', 'Pleural_Thickening': 'Routine'
}

print(f"\n{'Finding':<22} {'Tier':<10} {'AUC':<8} {'F1':<8} {'Precision':<12} {'Recall':<8} {'Support':<8}")
print("-" * 85)

per_class_auc = []
for i, label in enumerate(LABELS):
    auc = roc_auc_score(test_targets[:, i], test_probs[:, i])
    per_class_auc.append(auc)
    f1 = f1_score(test_targets[:, i], test_pred[:, i], zero_division=0)
    prec = precision_score(test_targets[:, i], test_pred[:, i], zero_division=0)
    rec = recall_score(test_targets[:, i], test_pred[:, i], zero_division=0)
    support = int(test_targets[:, i].sum())
    tier = triage_map[label]
    print(f"{label:<22} {tier:<10} {auc:<8.4f} {f1:<8.4f} {prec:<12.4f} {rec:<8.4f} {support:<8}")

print("-" * 85)

mean_auc = np.mean(per_class_auc)
critical_auc = np.mean([auc for label, auc in zip(LABELS, per_class_auc) if label in CRITICAL])
micro_f1 = f1_score(test_targets, test_pred, average='micro')
macro_f1 = f1_score(test_targets, test_pred, average='macro')

print(f"\nModel: {MODEL_LABEL}")
print(f"Mean Test AUC:     {mean_auc:.4f}")
print(f"Critical Test AUC: {critical_auc:.4f}")
print(f"Micro F1:          {micro_f1:.4f}")
print(f"Macro F1:          {macro_f1:.4f}")

## Save results

In [ ]:
import json

results = {
    'model': MODEL_LABEL,
    'checkpoint': CHECKPOINT_PATH,
    'checkpoint_epoch': checkpoint['epoch'] + 1,
    'mean_test_auc': float(mean_auc),
    'critical_test_auc': float(critical_auc),
    'micro_f1': float(micro_f1),
    'macro_f1': float(macro_f1),
    'thresholds': optimal_thresholds,
    'per_class': {
        label: {
            'auc': float(roc_auc_score(test_targets[:, i], test_probs[:, i])),
            'f1': float(f1_score(test_targets[:, i], test_pred[:, i], zero_division=0)),
            'precision': float(precision_score(test_targets[:, i], test_pred[:, i], zero_division=0)),
            'recall': float(recall_score(test_targets[:, i], test_pred[:, i], zero_division=0)),
            'support': int(test_targets[:, i].sum()),
            'tier': triage_map[label]
        }
        for i, label in enumerate(LABELS)
    }
}

with open('D:/cxr-triage/notebooks/focal_no_clahe_final_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Saved to focal_no_clahe_final_results.json")